In [13]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [14]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [15]:
# BB84 Quantum Key Distribution : simulation WITH an attacker (Eve)
#
# Eve intercepts every qubit, measures it in a random basis, then
# re-prepares and forwards a new qubit.  Because she guesses the
# basis correctly only ~50% of the time, she disturbs ~25% of the
# sifted bits : enough for Alice and Bob to detect her.

In [16]:
# Quantum Random Number Generator
# Each qubit is prepared in (|0> + |1>) / sqrt(2) by applying H to |0>.
# Measuring this state yields 0 or 1 with equal probability — true quantum randomness.

def quantum_random_bits(n):
    """Return n random bits from quantum measurement of n H|0> qubits."""
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)              # H|0> = (|0> + |1>) / sqrt(2)
    qc.measure(range(n), range(n))

    simulator = BasicSimulator()
    compiled  = transpile(qc, simulator)
    result    = simulator.run(compiled, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]
    # Qiskit is little-endian: rightmost char = qubit 0, so reverse.
    return [int(b) for b in reversed(bitstring)]

In [17]:
# ALICE
# (Identical to the plain protocol, Alice is unaware of Eve.)

N = 20   # number of qubits transmitted

alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)  # 0 = rectilinear, 1 = diagonal

print("=== ALICE ===")
print(f"Bits:  {alice_bits}")
print(f"Bases: {alice_bases}  (0=+, 1=x)")

=== ALICE ===
Bits:  [0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1]
Bases: [1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1]  (0=+, 1=x)


In [18]:
# EVE  (attacker : intercepts the quantum channel)

# Eve picks random measurement bases, measures each qubit
# (collapsing its state), then re-prepares and forwards a new
# qubit in the state she observed.
#
# When Eve's basis matches Alice's: her copy is correct, no disturbance.
# When they differ: Eve sends the wrong state 50% of the time,
# introducing a detectable error at Bob's end.
#
# Circuit A: Alice prepares, Eve measures (in her basis)
# Eve then re-encodes based on her result and sends to Bob.

eve_bases = quantum_random_bits(N)

print("=== EVE (secret) ===")
print(f"Bases: {eve_bases}  (0=+, 1=x)")
print()

# --- Circuit A: Alice -> Eve ---
qc_eve = QuantumCircuit(N, N)

for i in range(N):
    # Alice's encoding
    if alice_bits[i] == 1:
        qc_eve.x(i)
    if alice_bases[i] == 1:
        qc_eve.h(i)
    # Eve rotates into her measurement basis before measuring
    if eve_bases[i] == 1:
        qc_eve.h(i)

qc_eve.measure(range(N), range(N))

simulator = BasicSimulator()
compiled  = transpile(qc_eve, simulator)
result    = simulator.run(compiled, shots=1).result()
bitstring = list(result.get_counts().keys())[0]
eve_results = [int(b) for b in reversed(bitstring)]

print(f"Eve's measured bits (secret): {eve_results}")
print()
# Show how many bases Eve guessed correctly
eve_correct = sum(e == a for e, a in zip(eve_bases, alice_bases))
print(f"Eve's basis matched Alice's : {eve_correct}/{N} ({eve_correct/N:.0%})")
print("(Eve cannot know this)")

=== EVE (secret) ===
Bases: [1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1]  (0=+, 1=x)

Eve's measured bits (secret): [0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1]

Eve's basis matched Alice's : 14/20 (70%)
(Eve cannot know this)


In [19]:
# QUANTUM CHANNEL (Eve -> Bob)  to  BOB

# Eve re-prepares each qubit in *her* measured basis and sends to Bob.
# Bob measures in his own randomly chosen basis.
#
# Circuit B: Eve re-encodes (Bob decodes) measure

bob_bases = quantum_random_bits(N)

print("=== BOB ===")
print(f"Bases: {bob_bases}  (0=+, 1=x)")
print()

# --- Circuit B: Eve -> Bob ---
qc_bob = QuantumCircuit(N, N)

for i in range(N):
    # Eve re-encodes in her basis using her measurement result
    if eve_results[i] == 1:
        qc_bob.x(i)
    if eve_bases[i] == 1:
        qc_bob.h(i)
    # Bob decodes in his chosen basis
    if bob_bases[i] == 1:
        qc_bob.h(i)

qc_bob.measure(range(N), range(N))

simulator = BasicSimulator()
compiled  = transpile(qc_bob, simulator)
result    = simulator.run(compiled, shots=1).result()
bitstring = list(result.get_counts().keys())[0]
bob_results = [int(b) for b in reversed(bitstring)]

print(f"Bob's measured bits: {bob_results}")

=== BOB ===
Bases: [0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1]  (0=+, 1=x)

Bob's measured bits: [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1]


In [20]:
# CLASSICAL CHANNEL : Basis Comparison  (public)

matching = [i for i in range(N) if alice_bases[i] == bob_bases[i]]

alice_sifted = [alice_bits[i]  for i in matching]
bob_sifted   = [bob_results[i] for i in matching]

print("=== BASIS COMPARISON ===")
header = f"{'Qubit':>5} | {'Alice':>6} | {'Bob':>6} | {'Keep?':>6}"
print(header)
print("-" * len(header))
for i in range(N):
    keep = "YES" if alice_bases[i] == bob_bases[i] else ""
    print(f"{i:>5} | {alice_bases[i]:>6} | {bob_bases[i]:>6} | {keep:>6}")

print(f"\nMatched {len(matching)} / {N} positions: {matching}")
print(f"\nAlice sifted key : {alice_sifted}")
print(f"Bob   sifted key : {bob_sifted}")

=== BASIS COMPARISON ===
Qubit |  Alice |    Bob |  Keep?
--------------------------------
    0 |      1 |      0 |       
    1 |      1 |      0 |       
    2 |      1 |      1 |    YES
    3 |      0 |      0 |    YES
    4 |      1 |      0 |       
    5 |      1 |      0 |       
    6 |      0 |      0 |    YES
    7 |      1 |      1 |    YES
    8 |      1 |      0 |       
    9 |      1 |      1 |    YES
   10 |      0 |      0 |    YES
   11 |      1 |      1 |    YES
   12 |      0 |      0 |    YES
   13 |      1 |      1 |    YES
   14 |      1 |      1 |    YES
   15 |      0 |      0 |    YES
   16 |      0 |      0 |    YES
   17 |      1 |      1 |    YES
   18 |      0 |      1 |       
   19 |      1 |      1 |    YES

Matched 14 / 20 positions: [2, 3, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19]

Alice sifted key : [0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1]
Bob   sifted key : [0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1]


In [21]:
# ATTACK DETECTION

# Alice and Bob sacrifice a portion of their sifted key bits by
# comparing them publicly.  A non-zero error rate exposes Eve.
#
# Expected error rate WITH Eve intercepting all qubits: ~25%
#   (Eve wrong basis 50% of time, causes 50% error when wrong -> 25% overall)
# Expected error rate WITHOUT attacker: 0%
# Detection threshold: 11% (well below expected Eve noise of 25%)

THRESHOLD = 0.11

errors     = sum(a != b for a, b in zip(alice_sifted, bob_sifted))
error_rate = errors / len(alice_sifted) if alice_sifted else 0.0

print("=== ATTACK DETECTION ===")
print(f"Sifted key length : {len(alice_sifted)} bits (all compared publicly)")
print(f"Errors found      : {errors}")
print(f"Error rate        : {error_rate:.1%}")
print(f"Detection threshold: {THRESHOLD:.0%}")
print()
if error_rate > THRESHOLD:
    print("[ATTACK DETECTED] Error rate exceeds threshold.")
    print("Alice and Bob abort the protocol: the channel is compromised.")
else:
    print("[No attack detected] Error rate within acceptable bounds.")
    print("(With only 20 qubits Eve may get lucky: increase N for reliability)")
print()

# Summary for educational clarity
print("--- Summary ---")
print(f"Alice's bases : {alice_bases}")
print(f"Eve's bases   : {eve_bases}")
print(f"Bob's bases   : {bob_bases}")
print()
print(f"Alice bits    : {alice_bits}")
print(f"Eve results   : {eve_results}  (secret)")
print(f"Bob results   : {bob_results}")

=== ATTACK DETECTION ===
Sifted key length : 14 bits (all compared publicly)
Errors found      : 2
Error rate        : 14.3%
Detection threshold: 11%

[ATTACK DETECTED] Error rate exceeds threshold.
Alice and Bob abort the protocol: the channel is compromised.

--- Summary ---
Alice's bases : [1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1]
Eve's bases   : [1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1]
Bob's bases   : [0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1]

Alice bits    : [0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1]
Eve results   : [0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1]  (secret)
Bob results   : [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1]


In [22]:
# OTHER ATTACK: selective intercept-resend
#
# Eve now tries to be stealthier. Instead of measuring every qubit, she uses
# quantum randomness to choose only some qubits to intercept.
#
# Attack decision rule:
#   Generate two quantum-random mask bits per qubit.
#   Eve attacks unless both bits are 0.
# This attacks about 75% of qubits without using Python's random module.
#
# Expected effect:
#   Full intercept-resend disrupts about 25% of the sifted key.
#   This selective attack disrupts about 0.75 * 25% = 18.75%.
# Eve learns less key material, but also creates less noise.

ATTACK_2_N = 80
ATTACK_2_THRESHOLD = 0.11


def quantum_random_bits_chunked(n, chunk_size=20):
    """Return n quantum-random bits without building one huge circuit."""
    bits = []
    while len(bits) < n:
        take = min(chunk_size, n - len(bits))
        bits.extend(quantum_random_bits(take))
    return bits


attack2_simulator = BasicSimulator()


def measure_single_qubit(configure_circuit):
    """Build, run, and measure one qubit after configure_circuit adds gates."""
    qc = QuantumCircuit(1, 1)
    configure_circuit(qc)
    qc.measure(0, 0)

    compiled = transpile(qc, attack2_simulator)
    result = attack2_simulator.run(compiled, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]
    return int(bitstring[-1])


def prepare_alice_qubit(qc, bit, basis):
    """Alice prepares one BB84 qubit: basis 0 is +, basis 1 is x."""
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)


# --- ALICE, second demonstration ---
alice2_bits = quantum_random_bits_chunked(ATTACK_2_N)
alice2_bases = quantum_random_bits_chunked(ATTACK_2_N)

print("=== SECOND ATTACK: SELECTIVE INTERCEPT-RESEND ===")
print("=== ALICE 2 ===")
print(f"Number of qubits: {ATTACK_2_N}")
print(f"Bits preview : {alice2_bits[:24]} ...")
print(f"Bases preview: {alice2_bases[:24]} ...  (0=+, 1=x)")
print()


# --- EVE, second attack ---
# Eve attacks a QRNG-selected subset. The mask is secret.
eve2_mask_a = quantum_random_bits_chunked(ATTACK_2_N)
eve2_mask_b = quantum_random_bits_chunked(ATTACK_2_N)
eve2_attacks = [1 if (a == 1 or b == 1) else 0 for a, b in zip(eve2_mask_a, eve2_mask_b)]
eve2_bases = quantum_random_bits_chunked(ATTACK_2_N)
eve2_results = [None] * ATTACK_2_N

for i in range(ATTACK_2_N):
    if eve2_attacks[i] == 1:
        def eve_measurement(qc, i=i):
            prepare_alice_qubit(qc, alice2_bits[i], alice2_bases[i])
            if eve2_bases[i] == 1:
                qc.h(0)

        eve2_results[i] = measure_single_qubit(eve_measurement)

attacked_count = sum(eve2_attacks)

print("=== EVE 2 (secret) ===")
print(f"Attack mask preview : {eve2_attacks[:24]} ...  (1=intercept, 0=leave alone)")
print(f"Eve bases preview   : {eve2_bases[:24]} ...  (0=+, 1=x)")
print(f"Eve attacked        : {attacked_count}/{ATTACK_2_N} qubits ({attacked_count / ATTACK_2_N:.1%})")
print()


# --- BOB, second demonstration ---
# If Eve does not attack a qubit, Alice's original state is sent to Bob.
# If Eve attacks, Bob receives Eve's re-prepared qubit instead.
bob2_bases = quantum_random_bits_chunked(ATTACK_2_N)
bob2_results = []

for i in range(ATTACK_2_N):
    def bob_measurement(qc, i=i):
        if eve2_attacks[i] == 1:
            # Eve re-prepares using her measurement result and basis.
            if eve2_results[i] == 1:
                qc.x(0)
            if eve2_bases[i] == 1:
                qc.h(0)
        else:
            # No attack: the original Alice qubit reaches Bob.
            prepare_alice_qubit(qc, alice2_bits[i], alice2_bases[i])

        if bob2_bases[i] == 1:
            qc.h(0)

    bob2_results.append(measure_single_qubit(bob_measurement))

print("=== BOB 2 ===")
print(f"Bases preview  : {bob2_bases[:24]} ...  (0=+, 1=x)")
print(f"Results preview: {bob2_results[:24]} ...")
print()


# --- CLASSICAL CHANNEL, second demonstration ---
matching2 = [i for i in range(ATTACK_2_N) if alice2_bases[i] == bob2_bases[i]]
alice2_sifted = [alice2_bits[i] for i in matching2]
bob2_sifted = [bob2_results[i] for i in matching2]

print("=== BASIS COMPARISON 2 ===")
print(f"Matched {len(matching2)} / {ATTACK_2_N} positions")
print(f"First matched positions: {matching2[:24]}{' ...' if len(matching2) > 24 else ''}")
print(f"Alice sifted preview: {alice2_sifted[:24]}{' ...' if len(alice2_sifted) > 24 else ''}")
print(f"Bob sifted preview  : {bob2_sifted[:24]}{' ...' if len(bob2_sifted) > 24 else ''}")
print()


# --- ATTACK DETECTION, second demonstration ---
errors2 = sum(a != b for a, b in zip(alice2_sifted, bob2_sifted))
error_rate2 = errors2 / len(alice2_sifted) if alice2_sifted else 0.0

# Eve only has reliable information on sifted positions where she attacked
# and happened to choose the same basis as Alice and Bob.
eve2_known_positions = [
    i for i in matching2
    if eve2_attacks[i] == 1 and eve2_bases[i] == alice2_bases[i]
]
eve2_known_bits = [eve2_results[i] for i in eve2_known_positions]
eve2_known_rate = len(eve2_known_positions) / len(alice2_sifted) if alice2_sifted else 0.0

print("=== ATTACK DETECTION 2 ===")
print(f"Sifted key length      : {len(alice2_sifted)} bits")
print(f"Errors found           : {errors2}")
print(f"Error rate             : {error_rate2:.1%}")
print(f"Detection threshold    : {ATTACK_2_THRESHOLD:.0%}")
print(f"Eve reliable key bits  : {len(eve2_known_positions)} / {len(alice2_sifted)} ({eve2_known_rate:.1%})")
print(f"Eve known positions    : {eve2_known_positions[:24]}{' ...' if len(eve2_known_positions) > 24 else ''}")
print(f"Eve known bit preview  : {eve2_known_bits[:24]}{' ...' if len(eve2_known_bits) > 24 else ''}")
print()

if error_rate2 > ATTACK_2_THRESHOLD:
    print("[ATTACK DETECTED] Selective intercept-resend still caused too many errors.")
    print("Alice and Bob abort the protocol.")
else:
    print("[No attack detected] Eve's lower attack rate stayed under the threshold in this run.")
    print("This shows the trade-off: attacking fewer qubits gives Eve less information but may reduce detection risk.")


=== SECOND ATTACK: SELECTIVE INTERCEPT-RESEND ===
=== ALICE 2 ===
Number of qubits: 80
Bits preview : [0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0] ...
Bases preview: [1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0] ...  (0=+, 1=x)

=== EVE 2 (secret) ===
Attack mask preview : [1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1] ...  (1=intercept, 0=leave alone)
Eve bases preview   : [1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1] ...  (0=+, 1=x)
Eve attacked        : 65/80 qubits (81.2%)

=== BOB 2 ===
Bases preview  : [0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0] ...  (0=+, 1=x)
Results preview: [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1] ...

=== BASIS COMPARISON 2 ===
Matched 44 / 80 positions
First matched positions: [2, 3, 5, 7, 11, 12, 13, 15, 16, 17, 18, 21, 22, 23, 24, 26, 27, 31, 32, 34, 35, 36, 38, 40] ...
Alice sifted previe